In [1]:
import torch, torch.nn as nn, time
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.ToTensor()
train_ds = datasets.MNIST("/kaggle/working/data", train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST("/kaggle/working/data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

# ---- the two models, same task, different shape ----

mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)

cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),   # 28x28x1  -> 28x28x32
    nn.MaxPool2d(2),                             #          -> 14x14x32
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),  #          -> 14x14x64
    nn.MaxPool2d(2),                             #          -> 7x7x64
    nn.Flatten(),                                #          -> 3136
    nn.Linear(64*7*7, 128), nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

def count(m): return sum(p.numel() for p in m.parameters())

def run(model, name, epochs=3):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    print(f"\n{name}  ({count(model):,} parameters)")
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        for images, labels in train_dl:
            images, labels = images.to(device), labels.to(device)
            loss = loss_fn(model(images), labels)
            opt.zero_grad(); loss.backward(); opt.step()

        model.eval(); correct = 0
        with torch.no_grad():
            for images, labels in test_dl:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
        print(f"  epoch {epoch+1}  accuracy {correct/len(test_ds):.4f}  ({time.time()-t0:.1f}s)")

run(mlp, "MLP")
run(cnn, "CNN")

100%|██████████| 9.91M/9.91M [00:00<00:00, 58.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.64MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 13.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.9MB/s]



MLP  (203,530 parameters)
  epoch 1  accuracy 0.9476  (8.4s)
  epoch 2  accuracy 0.9624  (7.7s)
  epoch 3  accuracy 0.9710  (7.7s)

CNN  (421,642 parameters)
  epoch 1  accuracy 0.9772  (9.3s)
  epoch 2  accuracy 0.9812  (8.1s)
  epoch 3  accuracy 0.9870  (8.1s)
